In [1]:
from langchain_community.document_loaders import TextLoader

# Initialize the loader
loader = TextLoader('Data.txt', encoding="utf-8")

# Load documents
documents = loader.load()

C:\Users\Sarve\AppData\Local\Temp\ipykernel_14196\2965794907.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\Sarve\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
text_document = loader.load()
text_document

[Document(metadata={'source': 'Data.txt'}, page_content='This project sits at the intersection of power electronics, electrochemistry, and renewable energy integration.Electrolyzers—whether Proton Exchange Membrane (PEM) or Alkaline (AEL)—are non-linear, high-current, low-voltage DC loads. Feeding them directly from variable renewable sources (PV, wind) via standard AC/DC or non-optimized converters introduces severe power quality issues, thermal stress, and current ripple that degrades catalyst membranes.The primary technical objective is optimizing power converter efficiency while maintaining tight current control and minimal ripple across wide operating ranges.Key Electrical Requirements for Electrolyzer CouplingUltra-Low Current Ripple: Electrolyzer stacks suffer accelerated degradation and reduced Faradaic efficiency when subjected to high current ripple (typically requires $< 2\\text{--}5\\%$).High Conversion Efficiency ($>97\\%$ Target): Every 1% efficiency gain in a multi-megaw

In [3]:
def split_documents(docs, chunk_size=1000, chunk_overlap=200):
    chunks = []
    for doc in docs:
        text = getattr(doc, 'page_content', str(doc))
        start = 0
        L = len(text)
        while start < L:
            end = min(start + chunk_size, L)
            chunk_text = text[start:end]
            metadata = getattr(doc, 'metadata', {}) if hasattr(doc, 'metadata') else doc.get('metadata', {}) if isinstance(doc, dict) else {}
            chunks.append(type('Doc', (), {'page_content': chunk_text, 'metadata': metadata})())
            if end == L:
                break
            start = end - chunk_overlap if end - chunk_overlap > start else end
    return chunks

text_splitter = type('TS', (), {'split_documents': staticmethod(split_documents)})()


In [4]:
finalDocuments = text_splitter.split_documents(documents)
texts = [getattr(doc, 'page_content', '') for doc in finalDocuments]
texts

['This project sits at the intersection of power electronics, electrochemistry, and renewable energy integration.Electrolyzers—whether Proton Exchange Membrane (PEM) or Alkaline (AEL)—are non-linear, high-current, low-voltage DC loads. Feeding them directly from variable renewable sources (PV, wind) via standard AC/DC or non-optimized converters introduces severe power quality issues, thermal stress, and current ripple that degrades catalyst membranes.The primary technical objective is optimizing power converter efficiency while maintaining tight current control and minimal ripple across wide operating ranges.Key Electrical Requirements for Electrolyzer CouplingUltra-Low Current Ripple: Electrolyzer stacks suffer accelerated degradation and reduced Faradaic efficiency when subjected to high current ripple (typically requires $< 2\\text{--}5\\%$).High Conversion Efficiency ($>97\\%$ Target): Every 1% efficiency gain in a multi-megawatt green hydrogen plant translates to tons of saved el

In [5]:
##Huggingface embedding
import os
from dotenv import load_dotenv
load_dotenv()


False

In [6]:
hf = os.getenv('HF_Token')
if hf:
    os.environ['HF_Token'] = hf
else:
    print('HF_Token not set; proceeding without it')

HF_Token not set; proceeding without it


In [7]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

def make_embeddings(texts):
    return model.encode(texts, show_progress_bar=False).tolist()

embeddings = make_embeddings([t[:1000] for t in texts])
print('computed', len(embeddings), 'embeddings')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2878.67it/s]


computed 6 embeddings


In [8]:
# Prepare documents with id and metadata
prepared_docs = []
for i, doc in enumerate(finalDocuments):
    meta = getattr(doc, 'metadata', {})
    D = type('Doc', (), {})()
    D.page_content = getattr(doc, 'page_content', '')
    D.metadata = meta
    D.id = str(i)
    prepared_docs.append(D)

# recompute embeddings for prepared docs
texts = [d.page_content for d in prepared_docs]
embs = make_embeddings([t[:1000] for t in texts])

# Use faiss via langchain if available; else save embeddings to file
try:
    from langchain_community.vectorstores import FAISS
    db = FAISS.from_documents(prepared_docs, embs)
    print('FAISS DB created')
except Exception as e:
    import json
    with open('embeddings.json', 'w', encoding='utf-8') as f:
        json.dump({'docs': texts, 'embeddings': embs}, f)
    print('Saved embeddings to embeddings.json:', str(e))


Saved embeddings to embeddings.json: 'list' object has no attribute 'embed_documents'


In [9]:
try:
    from langchain_community.vectorstores import FAISS
    db = FAISS.from_documents(prepared_docs, embs)
    print('FAISS DB created')
except Exception as e:
    import numpy as np, types
    emb_matrix = np.array(embs)
    def _similarity_search_text(query, k=5):
        q_emb = make_embeddings([query])[0]
        sims = emb_matrix.dot(q_emb) / (np.linalg.norm(emb_matrix, axis=1) * (np.linalg.norm(q_emb) + 1e-12))
        idx = sims.argsort()[::-1][:k]
        return [prepared_docs[i] for i in idx]
    db = types.SimpleNamespace(similarity_search=_similarity_search_text)
    print('Fallback similarity search ready:', str(e))


Fallback similarity search ready: 'list' object has no attribute 'embed_documents'


In [10]:
query = input("Enter your question : ")
docs = db.similarity_search(query)
for i in range(len(docs)):
    print(docs[i].page_content)

ulti-Phase Interleaved Buck Converter (IBC)Simple control, extremely low output current ripple due to phase cancellation, high power density.Lacks galvanic isolation; limited voltage step-down ratio.Non-isolated low-voltage PV/DC bus to stack interfaces.Dual Active Bridge (DAB)Bidirectional power flow, inherent galvanic isolation, Zero Voltage Switching (ZVS) capability.High circulating currents at light loads without advanced phase-shift modulation.High-power DC microgrids and MVDC-to-electrolyzer interfaces.LLC Resonant / Two-Stage Hybrid (e.g., IBC + LLC)High efficiency across wide dynamic range, high output ripple suppression, soft-switching (ZVS/ZCS).Increased component count and complex frequency/phase control.Isolated high-efficiency modular power supplies.Interleaved Boost/Buck-BoostFlexible gain for wide PV array input ranges.Higher switch stress and thermal concentration.Direct PV-to-Electrolyzer interface.Core Performance Metrics to BenchmarkTo build a rigorous performance a